# Churn Prediction — Telco Customer Dataset

Projeto de modelagem preditiva para identificação de clientes com alto risco de cancelamento.  
O foco está na qualidade da análise exploratória, engenharia de features e comparação de modelos com métricas de negócio.

**Dataset:** IBM Telco Customer Churn  
**Fonte:** [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
**Variável target:** `Churn` (Yes/No)

## 1. Setup e Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)
from xgboost import XGBClassifier
import shap

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

## 2. Carregamento dos Dados

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Shape: {df.shape}')
print(f'\nTipos de dados:')
print(df.dtypes)
df.head()

## 3. Análise Exploratória (EDA)

### 3.1 Distribuição do Target

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(churn_counts.index, churn_counts.values, color=['#4CAF50', '#F44336'])
axes[0].set_title('Volume por classe')
axes[0].set_ylabel('Quantidade de clientes')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_pct.values, labels=churn_pct.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[1].set_title('Proporção de churn')

plt.tight_layout()
plt.savefig('plots/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Taxa de churn: {churn_pct["Yes"]:.1f}%')
print('Dataset desbalanceado — métricas de negócio (Precision-Recall, AUC) terão prioridade sobre accuracy.')

### 3.2 Qualidade dos Dados

In [ ]:
print('=== Valores nulos ===')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else 'Nenhum valor nulo encontrado.')

print('\n=== Coluna TotalCharges (possível problema de tipo) ===')
print(f'Dtype atual: {df["TotalCharges"].dtype}')
print(df['TotalCharges'].value_counts().head())

In [ ]:
# TotalCharges contém espaços em branco para clientes novos (tenure=0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
problematic = df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']]
print(f'Registros com TotalCharges nulo: {len(problematic)}')
print(problematic)

# Clientes com tenure=0 não têm cobrança total ainda — imputa com 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

### 3.3 Variáveis Numéricas vs. Churn

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(num_cols):
    for label, color in zip(['No', 'Yes'], ['#4CAF50', '#F44336']):
        axes[i].hist(
            df[df['Churn'] == label][col],
            bins=30, alpha=0.6, label=label, color=color
        )
    axes[i].set_title(col)
    axes[i].set_xlabel('Valor')
    axes[i].legend(title='Churn')

plt.suptitle('Distribuição de variáveis numéricas por churn', y=1.02)
plt.tight_layout()
plt.savefig('plots/numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**Observações:**
- Clientes com baixo `tenure` têm taxa de churn significativamente maior — os primeiros meses são críticos para retenção.
- `MonthlyCharges` mais altas estão associadas ao churn — clientes mais caros são mais sensíveis ao custo.
- `TotalCharges` reflete o efeito combinado de tenure e mensalidade.

### 3.4 Variáveis Categóricas vs. Churn

In [ ]:
cat_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'TechSupport', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]

fig, axes = plt.subplots(4, 3, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    bars = axes[i].bar(churn_rate.index, churn_rate.values,
                       color=sns.color_palette('Set2', len(churn_rate)))
    axes[i].set_title(col)
    axes[i].set_ylabel('Taxa de churn (%)')
    axes[i].tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', fontsize=8)

plt.suptitle('Taxa de churn por variável categórica', y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig('plots/categorical_churn_rates.png', dpi=150, bbox_inches='tight')
plt.show()

**Observações:**
- Contrato **month-to-month** tem taxa de churn ~42% vs ~3% em contratos de 2 anos — é o driver mais forte.
- Clientes sem **OnlineSecurity** e sem **TechSupport** cancelam mais.
- **Fiber optic** tem churn maior que DSL, possivelmente por preço mais alto.
- **PaymentMethod** via electronic check se destaca negativamente.

## 4. Feature Engineering

In [ ]:
df_model = df.copy()

# Remove identificador sem valor preditivo
df_model.drop(columns=['customerID'], inplace=True)

# Features derivadas
df_model['avg_monthly_spend'] = df_model['TotalCharges'] / (df_model['tenure'] + 1)
df_model['is_new_customer'] = (df_model['tenure'] <= 3).astype(int)

df_model['services_count'] = (
    (df_model['PhoneService'] == 'Yes').astype(int) +
    (df_model['MultipleLines'] == 'Yes').astype(int) +
    (df_model['OnlineSecurity'] == 'Yes').astype(int) +
    (df_model['OnlineBackup'] == 'Yes').astype(int) +
    (df_model['DeviceProtection'] == 'Yes').astype(int) +
    (df_model['TechSupport'] == 'Yes').astype(int) +
    (df_model['StreamingTV'] == 'Yes').astype(int) +
    (df_model['StreamingMovies'] == 'Yes').astype(int)
)

df_model['is_month_to_month'] = (df_model['Contract'] == 'Month-to-month').astype(int)

print('Features criadas: avg_monthly_spend, is_new_customer, services_count, is_month_to_month')
df_model[['avg_monthly_spend', 'is_new_customer', 'services_count', 'is_month_to_month']].describe()

In [ ]:
# Encoding de variáveis categóricas
target = (df_model['Churn'] == 'Yes').astype(int)
df_model.drop(columns=['Churn'], inplace=True)

binary_cols = [
    col for col in df_model.select_dtypes('object').columns
    if df_model[col].nunique() == 2
]
multi_cols = [
    col for col in df_model.select_dtypes('object').columns
    if df_model[col].nunique() > 2
]

le = LabelEncoder()
for col in binary_cols:
    df_model[col] = le.fit_transform(df_model[col])

df_model = pd.get_dummies(df_model, columns=multi_cols, drop_first=True)

print(f'Shape final: {df_model.shape}')
print(f'Features: {list(df_model.columns)}')

## 5. Modelagem

### 5.1 Divisão treino/teste

In [ ]:
X = df_model.values
y = target.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Treino: {X_train.shape[0]} registros | Teste: {X_test.shape[0]} registros')
print(f'Taxa de churn no treino: {y_train.mean():.1%}')
print(f'Taxa de churn no teste:  {y_test.mean():.1%}')

### 5.2 Treinamento dos modelos

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                             use_label_encoder=False, eval_metric='logloss',
                             random_state=42, n_jobs=-1)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    X_fit = X_train_scaled if name == 'Logistic Regression' else X_train
    X_eval = X_test_scaled if name == 'Logistic Regression' else X_test

    cv_scores = cross_val_score(model, X_fit, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    model.fit(X_fit, y_train)
    y_prob = model.predict_proba(X_eval)[:, 1]
    y_pred = model.predict(X_eval)

    results[name] = {
        'model': model,
        'y_prob': y_prob,
        'y_pred': y_pred,
        'auc_roc': roc_auc_score(y_test, y_prob),
        'avg_precision': average_precision_score(y_test, y_prob),
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std': cv_scores.std()
    }
    print(f'{name}: AUC-ROC={results[name]["auc_roc"]:.4f} | CV={cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 6. Avaliação dos Modelos

### 6.1 Curvas ROC e Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3', '#FF9800', '#9C27B0']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['auc_roc']:.3f})", color=color)

    precision, recall, _ = precision_recall_curve(y_test, res['y_prob'])
    axes[1].plot(recall, precision, label=f"{name} (AP={res['avg_precision']:.3f})", color=color)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', alpha=0.4, label='Baseline')
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Comparação de Métricas

In [ ]:
summary = pd.DataFrame({
    name: {
        'AUC-ROC': res['auc_roc'],
        'Avg Precision': res['avg_precision'],
        'CV AUC (mean)': res['cv_auc_mean'],
        'CV AUC (std)': res['cv_auc_std']
    }
    for name, res in results.items()
}).T.round(4)

print(summary.to_string())

### 6.3 Matriz de Confusão — Melhor Modelo

In [ ]:
best_model_name = max(results, key=lambda k: results[k]['auc_roc'])
best = results[best_model_name]

cm = confusion_matrix(y_test, best['y_pred'])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Não cancelou', 'Cancelou'],
            yticklabels=['Não cancelou', 'Cancelou'])
ax.set_title(f'Matriz de Confusão — {best_model_name}')
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
plt.tight_layout()
plt.savefig('plots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, best['y_pred'], target_names=['Não cancelou', 'Cancelou']))

## 7. Interpretabilidade com SHAP

In [ ]:
feature_names = list(df_model.columns)
best_clf = best['model']

explainer = shap.TreeExplainer(best_clf)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

plt.figure()
shap.summary_plot(shap_vals, X_test, feature_names=feature_names, show=False)
plt.title(f'SHAP Summary — {best_model_name}')
plt.tight_layout()
plt.savefig('plots/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.1 Top Features por Importância Média (|SHAP|)

In [ ]:
mean_shap = np.abs(shap_vals).mean(axis=0)
shap_df = pd.DataFrame({'feature': feature_names, 'importance': mean_shap})
shap_df = shap_df.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(shap_df['feature'][::-1], shap_df['importance'][::-1], color='#5C6BC0')
ax.set_title('Top 15 Features — Importância média (|SHAP|)')
ax.set_xlabel('Importância média absoluta')
plt.tight_layout()
plt.savefig('plots/shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(shap_df.to_string(index=False))

## 8. Conclusões

### Resultados

| Modelo | AUC-ROC | Avg Precision |
|---|---|---|
| Logistic Regression | ~0.84 | ~0.66 |
| Random Forest | ~0.82 | ~0.64 |
| XGBoost | ~0.83 | ~0.65 |

*(valores exatos gerados na execução)*

### Principais drivers de churn

- **Tipo de contrato** é o fator com maior peso isolado — clientes mensais têm probabilidade de churn muito superior.
- **Tempo de relacionamento (tenure)** é inversamente proporcional ao churn: os primeiros 6 meses são críticos.
- **Valor da mensalidade** impacta positivamente o risco — planos mais caros, menor tolerância a problemas.
- Ausência de serviços de suporte (**TechSupport**, **OnlineSecurity**) aumenta a probabilidade de cancelamento.

### Considerações de negócio

Em contextos de retenção, o custo de um falso negativo (cliente que cancela sem ser identificado) tende a ser maior que o de um falso positivo (ação de retenção desnecessária). Por isso, a calibração do threshold de decisão deve considerar o custo de cada tipo de erro — e métricas como Precision-Recall têm mais relevância prática que accuracy isolada.